In [ ]:
cd Q:\sachuriga\Sachuriga_Python\quattrocolo-nwb4fp\src

In [ ]:
from neurochat.nc_data import NData
from neurochat.nc_spike import NSpike
from neurochat.nc_spatial import NSpatial
import neurochat.nc_plot as nc_plot
from neurochat.nc_lfp import NLfp
import matplotlib.pyplot as plt
import numpy as np
from pynwb import NWBHDF5IO
import matplotlib.pyplot as plt
import numpy as np
import math
import pynapple as nap
import numpy as np
from scipy import signal
import matplotlib.pyplot as plt
import numpy as np
from sklearn.preprocessing import normalize

import sys
import nwb4fp.analyses.maps as mapp
from nwb4fp.analyses.examples.tracking_plot import plot_ratemap,plot_path,plot_ratemap_ax
from nwb4fp.analyses.fields import separate_fields_by_laplace, separate_fields_by_dilation,find_peaks,separate_fields_by_laplace_of_gaussian,calculate_field_centers,distance_to_edge_function, remove_fields_by_area, map_pass_to_unit_circle,which_field,compute_crossings
from elephant.statistics import time_histogram, instantaneous_rate
from nwb4fp.analyses import maps
from nwb4fp.analyses.data import pos2speed,speed_filtered_spikes,load_speed_fromNWB,load_units_fromNWB,find_run_indices
from nwb4fp.data.helpers import unit_location_ch
from scipy.ndimage import gaussian_filter
import ast
import pandas as pd

In [ ]:
df_files = pd.read_pickle(r'Q:\crhip\Sachuriga\filre_with_table\adjust_y_with _meanVAlue\clusters_with_tsneLabel/65588_2024-03-08_A_units_table_withDLC.pkl')
unit_table = pd.DataFrame(df_files)
pyramidal_df = unit_table[unit_table['cell_type']=="pyramidal"]
file_path_nwb = pyramidal_df['session_id'].iloc[0]
file_path_nwb

In [ ]:
import pandas as pd
import pandas as pd
file_path = r"Q:\sachuriga\Sachuriga_Python/quattrocolo-nwb4fp/ASSY-236-F.prb"

# Read the file and parse the dictionary
local_vars = {'np': np}
with open(file_path, 'r') as file:
    exec(file.read(), local_vars)  # Execute the file content with NumPy in scope

    
channel_groups = local_vars.get('channel_groups')
if channel_groups is None:
    raise ValueError(f"'channel_groups' not found in {file_path}")

# Assuming channel_groups is loaded from Step 1
data = []
for group_id, group_data in channel_groups.items():
    channels = group_data['channels']
    geometry = group_data['geometry']
    for channel in channels:
        x, y = geometry[channel]
        data.append({
            'group_id': group_id,
            'channel_id': channel,
            'x': x,
            'y': y
        })
probe_df = pd.DataFrame(data)

# 你的数据和 DataFrame
data = probe_df  # 我省略了完整数据，因为它已经在上文给出

# 分组函数
def group_channels_by_group(df, channels):
    grouped = {}
    for channel in channels:
        group = df[df['channel_id'] == channel]['group_id'].values[0]
        if group not in grouped:
            grouped[group] = []
        grouped[group].append(channel)
    return grouped

# 找到每个组的中间 channel_id
def find_middle_channel_per_group(df, grouped_channels):
    middle_channels = {}
    for group, channels in grouped_channels.items():
        group_df = df[df['channel_id'].isin(channels)]
        sorted_df = group_df.sort_values('y', ascending=False).reset_index(drop=True)
        middle_idx = len(sorted_df) // 2
        middle_channel = sorted_df.iloc[middle_idx]['channel_id']
        middle_channels[group] = middle_channel
    return middle_channels

# 映射中间 channel 到输入列表
def map_middle_channels_to_input(df, channel_list, middle_channels):
    output_list = []
    for channel in channel_list:
        group = df[df['channel_id'] == channel]['group_id'].values[0]
        output_list.append(middle_channels[group])
    return output_list

# 找到每个组中间 channel 的下第 4 个
def find_lower_four_channel_per_group(df, grouped_channels, middle_channels):
    lower_four_dict = {}
    for group, middle_channel in middle_channels.items():
        group_df = probe_df[probe_df['group_id'] == group][['channel_id', 'y']]
        sorted_group = group_df.sort_values('y', ascending=False).reset_index(drop=True)
        start_idx = sorted_group[sorted_group['channel_id'] == middle_channel].index[0]
        target_idx = start_idx + 4
        if target_idx < len(sorted_group):
            lower_four_dict[group] = sorted_group.iloc[target_idx]['channel_id']
        else:
            lower_four_dict[group] = np.nan
    return lower_four_dict

# 映射下第 4 个到输入列表
def map_lower_four_to_input(df, channel_list, lower_four_dict):
    lower_four_output_list = []
    for channel in channel_list:
        group = df[df['channel_id'] == channel]['group_id'].values[0]
        lower_four_output_list.append(lower_four_dict[group])
    return lower_four_output_list

channel_list = chs
# 执行分组和中间值计算
grouped_channels = group_channels_by_group(probe_df, channel_list)
middle_channels = find_middle_channel_per_group(probe_df, grouped_channels)
output_list = map_middle_channels_to_input(probe_df, channel_list, middle_channels)
lower_four_dict = find_lower_four_channel_per_group(probe_df, grouped_channels, middle_channels)
lower_four_output_list = map_lower_four_to_input(probe_df, channel_list, lower_four_dict)

# 输出结果
print("Grouped channels:", grouped_channels)
print("Middle channel per group:", middle_channels)
print("Output list (length 10):", output_list)
print("Lower four dict (by group):", lower_four_dict)
print("Lower four output list:", lower_four_output_list)

In [ ]:
chs_py = np.int64(list(middle_channels.values()))
chs_sr = np.int64([num for num in lower_four_dict.values() if not np.isnan(num)])
chs_sr

In [ ]:
i=0
channel_list = [11, 40,60,55]
median_list =  output_list
for ch in channel_list:
    plt.scatter(probe_df[probe_df['channel_id']==ch]['x'], probe_df[probe_df['channel_id']==ch]['y'], marker='o')  # Using scatter instead of plot to show dots
    plt.scatter(probe_df[probe_df['channel_id']==median_list[i]]['x'], probe_df[probe_df['channel_id']==median_list[i]]['y'], marker='^')
    plt.text(probe_df[probe_df['channel_id']==ch]['x'], probe_df[probe_df['channel_id']==ch]['y'], fr"median ch {median_list[i]}", fontsize=8, color='black')
    i+=1

In [ ]:
for i in range(64):
    ch = i
    plt.scatter(probe_df[probe_df['channel_id']==ch]['x'], probe_df[probe_df['channel_id']==ch]['y'], marker='o')  # Using scatter instead of plot to show dots
    #plt.scatter(probe_df[probe_df['channel_id']==median_list[i]]['x'], probe_df[probe_df['channel_id']==median_list[i]]['y'], marker='^')
    plt.text(probe_df[probe_df['channel_id']==ch]['x'], probe_df[probe_df['channel_id']==ch]['y'], fr"ch:{i}", fontsize=8, color='black')


In [ ]:
import pandas as pd
pd.set_option('display.max_rows', None)
np.set_printoptions(threshold=np.inf)
## unit11: unit_num = 8, ch = 13
filepath = rf"S:\Sachuriga\nwb\test4neo/{file_path_nwb}"
npdata = nap.load_file(filepath)
npdata

## Load data
pos_cord = load_speed_fromNWB(npdata['XY_mid_brain'])

## filter speed
raw_pos,combined_array, mask,speeds,smoothed_speed,filtered_speed = pos2speed(pos_cord[:,0], # times
                            pos_cord[:,1], # x
                            pos_cord[:,2], # y
                            filter_speed=True, 
                            min_speed = 0.05)

## filter spikes with speed
unit_num=2
raw_pos=combined_array
# ## filter spikes with speed
# spk = speed_filtered_spikes(spikes_time,
#                             pos_cord[:,0], # times
#                             mask)
#for i in range(40):
spikes_time = load_units_fromNWB(npdata['units'], unit_num = unit_num)
spk = speed_filtered_spikes(spikes_time,
                            raw_pos[:,0])
time_stemp = pos_cord[:,0]
fig = plt.figure()
ax = fig.add_subplot(111)
plot_ratemap_ax(raw_pos[:,1], # x
            raw_pos[:,2], # y
            raw_pos[:,0], # times
            spikes_time ,
            box_size=[1.0, 1.0], 
            bin_size=0.05,
            smoothing=0.1,ax=ax,plot=True)

x_input = npdata['units']['x'][unit_num]
y_input = npdata['units']['y'][unit_num]

In [ ]:
forward_ep = npdata["XY_mid_brain"]
RUN_interval = nap.IntervalSet(forward_ep.start, forward_ep.end)
eeg = npdata['lfp_raw']
lfp_times = npdata['lfp_times']
FS = 1000
wake_ep = npdata["XY_mid_brain"].time_support

ch_num = unit_location_ch(x=x_input,y=y_input)
eeg_time = lfp_times.restrict(RUN_interval)
eeg_example = eeg.restrict(RUN_interval)[:, ch_num]
pos_example = npdata["XY_mid_brain"].restrict(RUN_interval)

In [ ]:
import pynapple as nap
path = 'S:\Sachuriga/nwb/test4neo/65165_2023-07-10_15-08-59_A_phy_k_manual.nwb'


ch = 5

ndata = nap.load_file(path)
lfp = ndata['lfp_raw']

eeg_example = lfp[:,5]
fs=1000
# Step 1: Bandpass filter (0.1–125 Hz)
lowcut = 150   # Lower cutoff frequency in Hz
highcut = 250  # Upper cutoff frequency in Hz
nyquist = fs / 2  # Nyquist frequency
order = 4  # Filter order
b, a = signal.butter(order, [lowcut / nyquist, highcut / nyquist], btype='band')
bandpassed_data = signal.filtfilt(b, a, eeg_example.values)
t=eeg_example.index.values
sharp_waves = nap.Tsd (t=t,d = bandpassed_data)

plt.figure(figsize=(12, 6))
plt.plot(t, lfp_data, label='Original LFP', alpha=0.5)

In [ ]:
import numpy as np
from scipy import signal
from scipy.ndimage import gaussian_filter1d
import matplotlib.pyplot as plt

# Example: Generate synthetic LFP data (replace with your actual data)
fs = 1000  # Sampling frequency in Hz
original_lfp = eeg_example
t = eeg_example .t  # 1 second of data
lfp_data = eeg_example.d # 50 Hz signal + noise

# Step 1: Bandpass filter (0.1–125 Hz)
lowcut = 0.1   # Lower cutoff frequency in Hz
highcut = 125  # Upper cutoff frequency in Hz
nyquist = fs / 2  # Nyquist frequency
order = 4  # Filter order
b, a = signal.butter(order, [lowcut / nyquist, highcut / nyquist], btype='band')
bandpassed_data = signal.filtfilt(b, a, lfp_data)

# Step 2: Gaussian smoothing (σ = 10–20 ms)
sigma_ms = 10  # Sigma in milliseconds (adjust between 10–20 ms as needed)
sigma_samples = sigma_ms * fs / 1000  # Convert ms to samples (e.g., 15 ms * 1250 Hz / 1000 = 18.75 samples)
smoothed_data = gaussian_filter1d(bandpassed_data, sigma=sigma_samples)

lfp_smoothed =  nap.Tsd(t=t, d=bandpassed_data)

# Plot all stages: original, bandpassed, and smoothed
plt.figure(figsize=(12, 6))
plt.plot(t, lfp_data, label='Original LFP', alpha=0.5)
plt.plot(t, bandpassed_data, label='Bandpassed (0.1–125 Hz)', alpha=0.7)
plt.plot(t, smoothed_data, label=f'Gaussian Smoothed (σ = {sigma_ms} ms)', linewidth=2)
plt.xlabel('Time (s)')
plt.ylabel('Amplitude')
plt.legend()
plt.title('LFP Processing: Bandpass + Gaussian Smoothing')
plt.show()

# Optional: Plot frequency response of the bandpass filter
w, h = signal.freqz(b, a, fs=fs)
plt.figure(figsize=(10, 6))
plt.plot(w, 20 * np.log10(abs(h)), 'b')
plt.xlabel('Frequency (Hz)')
plt.ylabel('Amplitude (dB)')
plt.title('Bandpass Filter Frequency Response')
plt.grid()
plt.show()

In [ ]:
eeg_example =nap.Tsd(t=t, d=bandpassed_data)
fig = plt.figure(constrained_layout=True, figsize=(10, 6))
axd = fig.subplot_mosaic(
    [["ephys"], ["pos"]],
    height_ratios=[1, 0.4],
)

axd["ephys"].plot(eeg_example, label="CA1")
axd["ephys"].set_title("EEG (1250 Hz)")
axd["ephys"].set_ylabel("LFP (a.u.)")
axd["ephys"].set_xlabel("time (s)")
axd["ephys"].margins(0)
axd["ephys"].legend()
axd["pos"].plot(pos_example, color="black")
axd["pos"].margins(0)
axd["pos"].set_xlabel("time (s)")
axd["pos"].set_ylabel("Linearized Position")
axd["pos"].set_xlim(RUN_interval[0, 0], RUN_interval[0, 1])

In [ ]:
power = nap.compute_power_spectral_density(eeg_example, fs=1000, ep=wake_ep)

In [ ]:
fig, ax = plt.subplots(1, constrained_layout=True, figsize=(10, 4))
ax.plot(
    power[(power.index >= 1.0) & (power.index <= 100)],
    alpha=0.5,
    label="LFP Frequency Power",
)
ax.axvspan(6, 12, color="red", alpha=0.1)
ax.set_xlabel("Freq (Hz)")
ax.set_ylabel("Power/Frequency ")
ax.set_title("LFP Power spectral density")
ax.legend()

In [ ]:
pos_cord = load_speed_fromNWB(npdata['XY_mid_brain'])
raw_pos,combined_array, mask,speeds,smoothed_speed,filtered_speed = pos2speed(pos_cord[:,0], # times
                            pos_cord[:,1], # x
                            pos_cord[:,2], # y
                            filter_speed=True, 
                            min_speed = 0.05)


starts,stops = find_run_indices(smoothed_speed, threshold=0.02)
print("Starts:", time_stemp[starts])  # Output: Starts: [1, 6]
print("Stops:", time_stemp[stops])   # Output: Stops: [3, 7]

In [ ]:
raw_pos,combined_array, mask,speeds,smoothed_speed,filtered_speed = pos2speed(pos_cord[:,0], # times
                            pos_cord[:,1], # x
                            pos_cord[:,2], # y
                            filter_speed=True, 
                            min_speed = 0.05)
starts,stops = find_run_indices(smoothed_speed, threshold=0.05)
run_ep = nap.IntervalSet(start=time_stemp[starts], end=time_stemp[stops], time_units='s')

# The rest epoch is the data at all points where we do not have position data
rest_ep = wake_ep.set_diff(run_ep)

In [ ]:
power_run = nap.compute_mean_power_spectral_density(
    eeg_example, 1.5, fs=FS, ep=run_ep
)
power_rest = nap.compute_mean_power_spectral_density(
    eeg_example, 1.5, fs=FS, ep=rest_ep
)

In [ ]:
fig, ax = plt.subplots(1, constrained_layout=True, figsize=(10, 4))
ax.plot(
    power_run[(power_run.index >= 2.0) & (power_run.index <= 40)],
    alpha=1,
    label="Run",
    linewidth=2,
)
ax.plot(
    power_rest[(power_rest.index >= 2.0) & (power_rest.index <= 40)],
    alpha=1,
    label="Rest",
    linewidth=2,
)
ax.axvspan(6, 12, color="red", alpha=0.1)
ax.set_xlabel("Freq (Hz)")
ax.set_ylabel("Power/Frequency")
ax.set_title("LFP Fourier Decomposition")
ax.legend()

In [ ]:
fig, ax = plt.subplots(1, constrained_layout=True, figsize=(10, 4))
ax.plot(
    power_run[(power_run.index >= 2.0) & (power_run.index <= 40)],
    alpha=1,
    label="Run",
    linewidth=2,
)
ax.plot(
    power_rest[(power_rest.index >= 2.0) & (power_rest.index <= 40)],
    alpha=1,
    label="Rest",
    linewidth=2,
)
ax.axvspan(6, 12, color="red", alpha=0.1)
ax.set_xlabel("Freq (Hz)")
ax.set_ylabel("Power/Frequency")
ax.set_title("LFP Fourier Decomposition")
ax.legend()

In [ ]:
average_sampling_interval = np.median(np.diff(pos_cord[:,0]))
average_sampling_interval

In [ ]:
# We must define the frequency set that we'd like to use for our decomposition
freqs = np.geomspace(3, 250, 100)
mwt_RUN = nap.compute_wavelet_transform(eeg_example, fs=FS, freqs=freqs)
print(mwt_RUN)


In [ ]:
fig = plt.figure(constrained_layout=True, figsize=(10, 6))
gs = plt.GridSpec(3, 1, figure=fig, height_ratios=[1.0, 0.5, 0.1])

ax0 = plt.subplot(gs[0, 0])
pcmesh = ax0.pcolormesh(mwt_RUN.t, freqs, np.transpose(np.abs(mwt_RUN)))
ax0.grid(False)
ax0.set_yscale("log")
ax0.set_title("Wavelet Decomposition")
ax0.set_ylabel("Frequency (Hz)")
cbar = plt.colorbar(pcmesh, ax=ax0, orientation="vertical")
ax0.set_ylabel("Amplitude")

ax1 = plt.subplot(gs[1, 0], sharex=ax0)
ax1.plot(eeg_example)
ax1.set_ylabel("LFP (a.u.)")

ax1 = plt.subplot(gs[2, 0], sharex=ax0)
ax1.plot(pos_example, color="black")
ax1.set_xlabel("Time (s)")
ax1.set_ylabel("Pos.")

In [ ]:
ep_ex_rem = nap.IntervalSet(
    npdata["XY_mid_brain"].time_support['start'] + 97.0,
    npdata["XY_mid_brain"].time_support['start'] + 100.0,
)

raw_pos,combined_array, mask,speeds,smoothed_speed,filtered_speed = pos2speed(pos_cord[:,0], # times
                            pos_cord[:,1], # x
                            pos_cord[:,2], # y
                            filter_speed=True, 
                            min_speed = 0.05)
starts,stops = find_run_indices(smoothed_speed, threshold=0.05)
run_ep = nap.IntervalSet(start=time_stemp[starts], end=time_stemp[stops], time_units='s')

wake_ep = npdata["XY_mid_brain"].time_support


In [ ]:
forward_ep = npdata["XY_mid_brain"]
RUN_interval = nap.IntervalSet(forward_ep.start, forward_ep.end)
eeg = npdata['lfp_raw']
FS = 1250
wake_ep = npdata["XY_mid_brain"].time_support


In [ ]:
spikes = npdata["units"].restrict(run_ep)

In [ ]:
fig, ax = plt.subplots(1, constrained_layout=True, figsize=(10, 3))
ax.plot(eeg_example.restrict(ep_ex_rem))
ax.set_title("REM Local Field Potential")
ax.set_ylabel("LFP (a.u.)")
ax.set_xlabel("time (s)")
wake_ep['start']

In [ ]:
freqs = np.geomspace(5, 200, 25)

In [ ]:
cwt_rem = nap.compute_wavelet_transform(eeg_example.restrict(ep_ex_rem), fs=FS, freqs=freqs)

In [ ]:
# Define wavelet decomposition plotting function
def plot_timefrequency(freqs, powers, ax=None):
    im = ax.imshow(np.abs(powers), aspect="auto")
    ax.invert_yaxis()
    ax.set_xlabel("Time (s)")
    ax.set_ylabel("Frequency (Hz)")
    ax.get_xaxis().set_visible(False)
    ax.set(yticks=np.arange(len(freqs))[::2], yticklabels=np.rint(freqs[::2]))
    ax.grid(False)
    return im

fig = plt.figure(constrained_layout=True, figsize=(10, 6))
fig.suptitle("Wavelet Decomposition")
gs = plt.GridSpec(2, 1, figure=fig, height_ratios=[1.0, 0.3])

ax0 = plt.subplot(gs[0, 0])
im = plot_timefrequency(freqs, np.transpose(cwt_rem[:, :].values), ax=ax0)
cbar = fig.colorbar(im, ax=ax0, orientation="vertical")

ax1 = plt.subplot(gs[1, 0])
ax1.plot(eeg_example.restrict(ep_ex_rem))
ax1.set_ylabel("LFP (a.u.)")
ax1.set_xlabel("Time (s)")
ax1.margins(0)

In [ ]:
theta_band = nap.apply_bandpass_filter(eeg_example, cutoff=(6.0, 12.0), fs=FS)
plt.figure(constrained_layout=True, figsize=(12, 3))
plt.plot(eeg_example.restrict(ep_ex_rem), alpha=0.5)
plt.plot(theta_band.restrict(ep_ex_rem))
plt.xlabel("Time (s)")
plt.show()

In [ ]:
from scipy import signal

theta_phase = nap.Tsd(t=theta_band.t, d=np.angle(signal.hilbert(theta_band)))

In [ ]:
plt.figure(constrained_layout=True, figsize=(12, 3))
plt.subplot(211)
plt.plot(eeg_example.restrict(ep_ex_rem), alpha=0.5)
plt.plot(theta_band.restrict(ep_ex_rem))
plt.subplot(212)
plt.plot(theta_phase.restrict(ep_ex_rem), color='r')
plt.ylabel("Phase (rad)")
plt.xlabel("Time (s)")
plt.show()

In [ ]:
spikes = spikes[spikes.rate > 5.0]

In [ ]:
phase_modulation = nap.compute_1d_tuning_curves(
    group=spikes, feature=theta_phase, nb_bins=61, minmax=(-np.pi, np.pi)
)

In [ ]:
plt.figure(constrained_layout=True, figsize = (12, 3))
for i in range(11):
    plt.subplot(2,6,i+1)
    plt.plot(phase_modulation.iloc[:,i])
    plt.xlabel("Phase (rad)")
    plt.ylabel("Firing rate (Hz)")
plt.show()

In [ ]:
phase_modulation